In [1]:
from __future__ import annotations

import os
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from google.cloud import bigquery
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ID = os.getenv(
    "GCP_PROJECT",
    "pacey32-agency"
)

TEAM_TABLE = os.getenv(
    "TEAM_TABLE",
    "pacey32-agency.Team.TeamList"
)

CITY_REFERENCE_TABLE = os.getenv(
    "CITY_REFERENCE_TABLE",
    "pacey32-agency.City.CityReference"
)

GEOCODING_URL = (
    "https://geocoding-api.open-meteo.com/v1/search"
)

REQUEST_TIMEOUT = 60

GEOCODE_SLEEP_SECONDS = 0.25

FORCE_REFRESH = False

HEADERS = {
    "User-Agent": "pacey32-cityreference/1.0"
}

In [3]:
# ============================================================
# HTTP SESSION
# ============================================================

retry = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.5,
    status_forcelist=(429,500,502,503,504),
    allowed_methods=frozenset({"GET"})
)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))
session.headers.update(HEADERS)

BQ = bigquery.Client(project=PROJECT_ID)

In [4]:
# ============================================================
# SEARCH OVERRIDES
# ============================================================

SEARCH_OVERRIDES = {
    "St. Louis": "Saint Louis, Missouri",
    "St. Paul": "Saint Paul, Minnesota",
    "Paradise": "Paradise, Nevada",
    "Elmont": "Elmont, New York",
    "Sunrise": "Sunrise, Florida",
}

In [5]:
# ============================================================
# GEOCODER
# ============================================================

def geocode_city(city):

    search = SEARCH_OVERRIDES.get(city, city)

    response = session.get(
        GEOCODING_URL,
        params={
            "name": search,
            "count": 10,
            "language": "en",
            "format": "json"
        },
        timeout=REQUEST_TIMEOUT
    )

    response.raise_for_status()

    results = response.json().get("results", [])
    results = [
        r
        for r in results
        if r.get("country_code") in {"US","CA"}
    ]

    if not results:
        return {
            "city_name": city,
            "geocode_status":"NOT_FOUND"
        }

    results.sort(
        key=lambda x: x.get("population") or 0,
        reverse=True
    )

    best = results[0]

    latitude = best.get("latitude")
    longitude = best.get("longitude")

    population = best.get("population")
    if population is None:
        population = 0

    geography = None
    if latitude is not None and longitude is not None:
        geography = f"POINT({longitude} {latitude})"

    return {
        "city_name": city,
        "search_name": search,
        "state_province": best.get("admin1"),
        "country": best.get("country"),
        "country_code": best.get("country_code"),
        "latitude": latitude,
        "longitude": longitude,
        "geography": geography,
        "timezone": best.get("timezone"),
        "population": population,
        "elevation": best.get("elevation"),
        "open_meteo_id": best.get("id"),
        "geocode_status": "FOUND",
        "geocode_source": "Open-Meteo",
        "last_updated": datetime.now(timezone.utc)
    }

In [6]:
# ============================================================
# READ TEAM CITIES
# ============================================================

print("Reading distinct cities from TeamList...")

query = f"""
SELECT DISTINCT

    TRIM(venueLocation) AS city_name

FROM `{TEAM_TABLE}`

WHERE venueLocation IS NOT NULL
  AND TRIM(venueLocation) != ''

ORDER BY city_name
"""

query_job = BQ.query(query)

team_rows = [
    dict(row.items())
    for row in query_job.result()
]

team_df = pd.DataFrame(team_rows)

print(f"{len(team_df)} cities found.")

Reading distinct cities from TeamList...
32 cities found.


In [7]:
# ============================================================
# READ EXISTING CITYREFERENCE
# ============================================================

print("Reading existing CityReference...")

try:
    query = f"""
    SELECT DISTINCT
        city_name
    FROM `{CITY_REFERENCE_TABLE}`
    """

    query_job = BQ.query(query)

    existing_rows = [
        dict(row.items())
        for row in query_job.result()
    ]

    existing_df = pd.DataFrame(existing_rows)

    print(f"{len(existing_df)} cities already stored.")

except Exception:

    print("CityReference table not found.")

    existing_df = pd.DataFrame(
        columns=["city_name"]
    )

Reading existing CityReference...
CityReference table not found.


In [8]:
# ============================================================
# DETERMINE WHICH CITIES NEED GEOCODING
# ============================================================

if FORCE_REFRESH:

    due_df = team_df.copy()

else:

    due_df = team_df[
        ~team_df["city_name"].isin(
            existing_df["city_name"]
        )
    ].copy()

due_df = due_df.sort_values(
    "city_name"
).reset_index(drop=True)

print()

print(f"{len(due_df)} cities require geocoding.")


32 cities require geocoding.


In [9]:
# ============================================================
# GEOCODE ALL CITIES
# ============================================================

rows = []

for i, city in enumerate(due_df["city_name"], start=1):

    print(f"{i:>2}/{len(due_df)}  {city}")

    try:

        row = geocode_city(city)

    except Exception as exc:

        row = {
            "city_name": city,
            "search_name": SEARCH_OVERRIDES.get(city, city),
            "state_province": None,
            "country": None,
            "country_code": None,
            "latitude": None,
            "longitude": None,
            "geography": None,
            "timezone": None,
            "population": 0,
            "elevation": None,
            "open_meteo_id": None,
            "geocode_status": f"ERROR: {exc}",
            "geocode_source": "Open-Meteo",
            "last_updated": datetime.now(timezone.utc)
        }

    rows.append(row)

    time.sleep(GEOCODE_SLEEP_SECONDS)

city_reference_df = pd.DataFrame(rows)

 1/32  Anaheim
 2/32  Boston
 3/32  Buffalo
 4/32  Calgary
 5/32  Chicago
 6/32  Columbus
 7/32  Dallas
 8/32  Denver
 9/32  Detroit
10/32  Edmonton
11/32  Elmont
12/32  Los Angeles
13/32  Montreal
14/32  Nashville
15/32  New York
16/32  Newark
17/32  Ottawa
18/32  Paradise
19/32  Philadelphia
20/32  Pittsburgh
21/32  Raleigh
22/32  Salt Lake City
23/32  San Jose
24/32  Seattle
25/32  St. Louis
26/32  St. Paul
27/32  Sunrise
28/32  Tampa
29/32  Toronto
30/32  Vancouver
31/32  Washington
32/32  Winnipeg


In [10]:
# ============================================================
# VALIDATION
# ============================================================

print()
print("Summary")
print("-------")

print(city_reference_df["geocode_status"].value_counts())

print()

failed = city_reference_df[
    city_reference_df["geocode_status"] != "FOUND"
]

if len(failed):

    print("Cities requiring attention:")

    display(failed)

else:

    print("All cities geocoded successfully.")

display(city_reference_df)

print()
print(f"Generated {len(city_reference_df)} rows.")


Summary
-------
geocode_status
FOUND    32
Name: count, dtype: int64

All cities geocoded successfully.


,city_name,search_name,state_province,country,country_code,latitude,longitude,geography,timezone,population,elevation,open_meteo_id,geocode_status,geocode_source,last_updated
0,Anaheim,Anaheim,California,United States,US,33.83529,-117.91450,POINT(-117.9145 33.83529),America/Los_Angeles,350742,48.0,5323810,FOUND,Open-Meteo,2026-08-03 22:38:13.941514+00:00
1,Boston,Boston,Massachusetts,United States,US,42.35843,-71.05977,POINT(-71.05977 42.35843),America/New_York,653833,14.0,4930956,FOUND,Open-Meteo,2026-08-03 22:38:14.247202+00:00
2,Buffalo,Buffalo,New York,United States,US,42.88645,-78.87837,POINT(-78.87837 42.88645),America/New_York,258071,183.0,5110629,FOUND,Open-Meteo,2026-08-03 22:38:14.551233+00:00
3,Calgary,Calgary,Alberta,Canada,CA,51.05011,-114.08529,POINT(-114.08529 51.05011),America/Edmonton,1306784,1048.0,5913490,FOUND,Open-Meteo,2026-08-03 22:38:14.860692+00:00
4,Chicago,Chicago,Illinois,United States,US,41.85003,-87.65005,POINT(-87.65005 41.85003),America/Chicago,2664452,179.0,4887398,FOUND,Open-Meteo,2026-08-03 22:38:15.171021+00:00
5,Columbus,Columbus,Ohio,United States,US,39.96118,-82.99879,POINT(-82.99879 39.96118),America/New_York,913175,235.0,4509177,FOUND,Open-Meteo,2026-08-03 22:38:15.481781+00:00
6,Dallas,Dallas,Texas,United States,US,32.78306,-96.80667,POINT(-96.80667 32.78306),America/Chicago,1326087,128.0,4684888,FOUND,Open-Meteo,2026-08-03 22:38:15.786776+00:00
7,Denver,Denver,Colorado,United States,US,39.73915,-104.98470,POINT(-104.9847 39.73915),America/Denver,729019,1609.0,5419384,FOUND,Open-Meteo,2026-08-03 22:38:16.092269+00:00
8,Detroit,Detroit,Michigan,United States,US,42.33143,-83.04575,POINT(-83.04575 42.33143),America/Detroit,645705,183.0,4990729,FOUND,Open-Meteo,2026-08-03 22:38:16.402126+00:00
9,Edmonton,Edmonton,Alberta,Canada,CA,53.55014,-113.46871,POINT(-113.46871 53.55014),America/Edmonton,1010899,668.0,5946768,FOUND,Open-Meteo,2026-08-03 22:38:16.708966+00:00



Generated 32 rows.
